In [2]:
import sys

print("Python version:", sys.version)
print("AI News Agent project started!")

Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
AI News Agent project started!


In [3]:

%pip install feedparser requests pandas


Note: you may need to restart the kernel to use updated packages.


In [4]:

import feedparser
import requests
import pandas as pd
import json
import re
import hashlib
import unicodedata
import smtplib

from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime, timezone
from dateutil import parser as dateutil_parser
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, urlencode
from dataclasses import dataclass, field, asdict
from typing import Optional
from collections import Counter

print("All imports successful!")


All imports successful!


In [5]:
# ═══════════════════════════════════════════════════
#  🔧 CONFIGURATION — EDIT THESE VALUES
# ═══════════════════════════════════════════════════

# ── Gmail Settings ──
GMAIL_ADDRESS = "sankanoort@gmail.com"         # Your Gmail address
GMAIL_APP_PASSWORD = "sbqn duro pirw rfik"     # 16-char app password (NOT your login password)
RECIPIENT_EMAIL = "sankanoort@gmail.com"        # Where to send the digest (can be same as above)

# ── Ollama Settings ──
OLLAMA_BASE_URL = "http://localhost:11434"       # Default Ollama URL
OLLAMA_MODEL ="qwen3.5:0.8b"                     # Model to use

# ── Pipeline Settings ──
TOP_N_NEWS = 10                                  # Number of top news to email

print("⚙️  Configuration loaded.")
print(f"   Gmail sender:    {GMAIL_ADDRESS}")
print(f"   Recipient:       {RECIPIENT_EMAIL}")
print(f"   Ollama model:    {OLLAMA_MODEL}")
print(f"   Ollama URL:      {OLLAMA_BASE_URL}")
print(f"   Top N news:      {TOP_N_NEWS}")

⚙️  Configuration loaded.
   Gmail sender:    sankanoort@gmail.com
   Recipient:       sankanoort@gmail.com
   Ollama model:    qwen3.5:0.8b
   Ollama URL:      http://localhost:11434
   Top N news:      10


In [6]:

import requests
import time

print("Testing Ollama connection...")

try:
    start = time.time()

    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={
            "model": "qwen3.5:0.8b",
            "prompt": "Reply with only the word OK.",
            "stream": False,
            "keep_alive": "10m",
            "options": {
                "num_predict": 20
            }
        },
        timeout=(10, 300)
    )

    response.raise_for_status()
    result = response.json()

    print("Model response:", result.get("response", ""))
    print(f"Time taken: {time.time() - start:.1f} seconds")

except requests.exceptions.Timeout:
    print("Ollama is taking too long. Check model loading and system resources.")

except requests.exceptions.ConnectionError:
    print("Cannot connect to Ollama. Check whether Ollama is running.")

except requests.exceptions.RequestException as e:
    print("Ollama API error:", e)


Testing Ollama connection...
Model response: 
Time taken: 15.8 seconds


In [7]:
# ─────────────────────────────────────────────────
#  RSS FEED REGISTRY
# ─────────────────────────────────────────────────

RSS_SOURCES = [
    # ── Major International ──
    {
        "name": "BBC News - Top Stories",
        "url": "http://feeds.bbci.co.uk/news/rss.xml",
        "category": "general",
        "language": "en",
    },
    {
        "name": "CNN - Top Stories",
        "url": "http://rss.cnn.com/rss/edition.rss",
        "category": "general",
        "language": "en",
    },
    {
        "name": "Reuters - Top News",
        "url": "https://feeds.reuters.com/reuters/topNews",
        "category": "general",
        "language": "en",
    },
    {
        "name": "Al Jazeera",
        "url": "https://www.aljazeera.com/xml/rss/all.xml",
        "category": "general",
        "language": "en",
    },
    {
        "name": "The Guardian - World",
        "url": "https://www.theguardian.com/world/rss",
        "category": "general",
        "language": "en",
    },
    {
        "name": "NPR News",
        "url": "https://feeds.npr.org/1001/rss.xml",
        "category": "general",
        "language": "en",
    },

    # ── Technology ──
    {
        "name": "TechCrunch",
        "url": "https://techcrunch.com/feed/",
        "category": "technology",
        "language": "en",
    },
    {
        "name": "Ars Technica",
        "url": "https://feeds.arstechnica.com/arstechnica/index",
        "category": "technology",
        "language": "en",
    },
    {
        "name": "Wired",
        "url": "https://www.wired.com/feed/rss",
        "category": "technology",
        "language": "en",
    },

    # ── Business ──
    {
        "name": "CNBC Top News",
        "url": "https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=100003114",
        "category": "business",
        "language": "en",
    },

    # ── Science ──
    {
        "name": "NASA Breaking News",
        "url": "https://www.nasa.gov/rss/dyn/breaking_news.rss",
        "category": "science",
        "language": "en",
    },
]

print(f"📡 Registered {len(RSS_SOURCES)} RSS sources:")
for src in RSS_SOURCES:
    print(f"   • {src['name']:30s} [{src['category']}]")

📡 Registered 11 RSS sources:
   • BBC News - Top Stories         [general]
   • CNN - Top Stories              [general]
   • Reuters - Top News             [general]
   • Al Jazeera                     [general]
   • The Guardian - World           [general]
   • NPR News                       [general]
   • TechCrunch                     [technology]
   • Ars Technica                   [technology]
   • Wired                          [technology]
   • CNBC Top News                  [business]
   • NASA Breaking News             [science]


In [8]:
# ─────────────────────────────────────────────────
#  FETCH: Pull raw data from ALL RSS sources
# ─────────────────────────────────────────────────

def fetch_rss_raw(source: dict, timeout: int = 15) -> dict:
    """Fetch raw RSS feed data from a single source."""
    result = {
        "source": source,
        "feed_meta": {},
        "raw_entries": [],
        "status": "ok",
        "error": None,
    }
    
    try:
        feed = feedparser.parse(source["url"])
        
        if hasattr(feed, 'status') and feed.status >= 400:
            result["status"] = "error"
            result["error"] = f"HTTP {feed.status}"
            return result
        
        if feed.bozo and not feed.entries:
            result["status"] = "error"
            result["error"] = str(feed.bozo_exception)
            return result
        
        result["feed_meta"] = {
            "title": feed.feed.get("title", ""),
            "link": feed.feed.get("link", ""),
            "language": feed.feed.get("language", ""),
            "updated": feed.feed.get("updated", ""),
        }
        
        result["raw_entries"] = [
            dict(entry) for entry in feed.entries[:15]
        ]
        
    except Exception as e:
        result["status"] = "error"
        result["error"] = str(e)
    
    return result


# ── Fetch from all RSS sources ──
print("📡 Fetching raw RSS data...\n")

raw_rss_data = []
for source in RSS_SOURCES:
    result = fetch_rss_raw(source)
    raw_rss_data.append(result)
    
    icon = "✅" if result["status"] == "ok" else "❌"
    count = len(result["raw_entries"])
    err = f" — {result['error']}" if result["error"] else ""
    print(f"   {icon} {source['name']:30s} → {count:3d} entries{err}")

total_raw = sum(len(r["raw_entries"]) for r in raw_rss_data)
print(f"\n📊 Total raw entries fetched: {total_raw}")

📡 Fetching raw RSS data...

   ✅ BBC News - Top Stories         →  15 entries
   ✅ CNN - Top Stories              →  15 entries
   ❌ Reuters - Top News             →   0 entries — <urlopen error [Errno 11001] getaddrinfo failed>
   ✅ Al Jazeera                     →  15 entries
   ✅ The Guardian - World           →  15 entries
   ✅ NPR News                       →  10 entries
   ✅ TechCrunch                     →  15 entries
   ✅ Ars Technica                   →  15 entries
   ✅ Wired                          →  15 entries
   ✅ CNBC Top News                  →  15 entries
   ✅ NASA Breaking News             →  10 entries

📊 Total raw entries fetched: 140


In [9]:
# ─────────────────────────────────────────────────
#  DATA MODEL: Unified article structure
# ─────────────────────────────────────────────────

@dataclass
class Article:
    """Unified article structure across all source types."""
    title: str = ""
    url: str = ""
    source_name: str = ""
    source_type: str = ""
    description: str = ""
    content: str = ""
    author: str = ""
    published: str = ""
    image_url: str = ""
    categories: list = field(default_factory=list)

    def __repr__(self):
        return f"Article('{self.title[:50]}...', source='{self.source_name}')"


print("✅ Article dataclass defined.")

✅ Article dataclass defined.


In [10]:
# ─────────────────────────────────────────────────
#  PARSER: RSS raw entry → Article
# ─────────────────────────────────────────────────

def parse_rss_entry(entry: dict, source_name: str) -> Article:
    """Parse a single raw RSS entry into a unified Article."""
    article = Article()
    article.source_name = source_name
    article.source_type = "rss"
    
    article.title = entry.get("title", "No Title")
    article.url = entry.get("link", entry.get("id", ""))
    article.description = entry.get("summary", entry.get("description", ""))
    
    content_list = entry.get("content", [])
    if content_list and isinstance(content_list, list):
        article.content = content_list[0].get("value", "")
    else:
        article.content = article.description
    
    article.author = entry.get("author", "")
    if not article.author:
        authors = entry.get("authors", [])
        if authors:
            article.author = ", ".join(a.get("name", "") for a in authors)
    
    raw_date = entry.get("published", entry.get("updated", ""))
    article.published = raw_date
    
    media_thumbs = entry.get("media_thumbnail", [])
    if media_thumbs and isinstance(media_thumbs, list):
        article.image_url = media_thumbs[0].get("url", "")
    if not article.image_url:
        media_content = entry.get("media_content", [])
        if media_content and isinstance(media_content, list):
            for mc in media_content:
                if "image" in mc.get("type", "") or mc.get("medium") == "image":
                    article.image_url = mc.get("url", "")
                    break
    if not article.image_url:
        enclosures = entry.get("enclosures", entry.get("links", []))
        for enc in enclosures:
            if "image" in enc.get("type", ""):
                article.image_url = enc.get("href", enc.get("url", ""))
                break
    
    tags = entry.get("tags", [])
    if tags:
        article.categories = [t.get("term", "") for t in tags if t.get("term")]
    
    return article


# ── Parse ALL RSS entries ──
print("🔍 Parsing RSS entries into Articles...\n")

parsed_articles = []

for rss_data in raw_rss_data:
    if rss_data["status"] != "ok":
        continue
    
    source_name = rss_data["source"]["name"]
    for raw_entry in rss_data["raw_entries"]:
        article = parse_rss_entry(raw_entry, source_name)
        parsed_articles.append(article)

print(f"✅ Parsed {len(parsed_articles)} articles from RSS")

if parsed_articles:
    sample = parsed_articles[0]
    print("\n📌 Sample parsed article:")
    for fld in ['title', 'url', 'source_name', 'source_type', 'author',
                'published', 'image_url', 'categories']:
        val = getattr(sample, fld)
        print(f"   {fld:15s} → {str(val)[:100]}")
    print(f"   {'description':15s} → {sample.description[:100]}...")

🔍 Parsing RSS entries into Articles...

✅ Parsed 140 articles from RSS

📌 Sample parsed article:
   title           → Trump announces deal for Russian diesel as Zelensky calls it a gift to Putin
   url             → https://www.bbc.co.uk/news/articles/cm1dwgr666wno?at_medium=RSS&at_campaign=rss
   source_name     → BBC News - Top Stories
   source_type     → rss
   author          → 
   published       → Sat, 10 Oct 2026 04:38:20 GMT
   image_url       → https://ichef.bbci.co.uk/ace/standard/240/cpsprodpb/8c49/live/850032c0-c470-11f1-94f2-dbc7ff6db92b.j
   categories      → []
   description     → Ukraine's president sharply criticised the move, calling it an "investment in war that must be ended...


In [11]:
# ─────────────────────────────────────────────────
#  TEXT CLEANERS
# ─────────────────────────────────────────────────

def strip_html_tags(text: str) -> str:
    if not text:
        return ""
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text(separator=" ", strip=True)


def normalize_unicode(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    replacements = {
        "\u2018": "'", "\u2019": "'",
        "\u201c": '"', "\u201d": '"',
        "\u2013": "–", "\u2014": "—",
        "\u2026": "...", "\xa0": " ",
        "\u200b": "", "\ufeff": "",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def collapse_whitespace(text: str) -> str:
    if not text:
        return ""
    return re.sub(r'\s+', ' ', text).strip()


def remove_tracking_params(url: str) -> str:
    if not url:
        return ""
    tracking_params = {
        "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
        "ref", "source", "fbclid", "gclid", "msclkid",
    }
    parsed = urlparse(url)
    if not parsed.query:
        return url
    params = parse_qs(parsed.query, keep_blank_values=False)
    clean_params = {k: v for k, v in params.items() if k.lower() not in tracking_params}
    clean_query = urlencode(clean_params, doseq=True)
    return parsed._replace(query=clean_query).geturl()


def remove_api_truncation(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r'\[\+\d+ chars?\]', '', text)
    patterns = [
        r'\(Read more\.{0,3}\)',
        r'Continue reading\.{0,3}$',
        r'Read the full (?:article|story)\.{0,3}$',
        r'\.\.\.$ ',
    ]
    for pattern in patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    return text.strip()


def normalize_date(raw_date: str) -> str:
    if not raw_date or not raw_date.strip():
        return ""
    try:
        dt = dateutil_parser.parse(raw_date)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    except (ValueError, TypeError):
        return raw_date


def clean_article(article: Article) -> Article:
    """Apply all cleaning transformations to an Article."""
    article.title = collapse_whitespace(normalize_unicode(strip_html_tags(article.title)))
    article.description = remove_api_truncation(collapse_whitespace(normalize_unicode(strip_html_tags(article.description))))
    article.content = remove_api_truncation(collapse_whitespace(normalize_unicode(strip_html_tags(article.content))))
    article.author = collapse_whitespace(strip_html_tags(article.author))
    article.url = remove_tracking_params(article.url.strip())
    article.published = normalize_date(article.published)
    article.image_url = article.image_url.strip() if article.image_url else ""
    article.categories = [cat.strip() for cat in article.categories if cat and cat.strip()]
    return article


print("✅ All cleaning functions defined.")

✅ All cleaning functions defined.


In [12]:
# ─────────────────────────────────────────────────
#  RUN CLEANING on all parsed articles
# ─────────────────────────────────────────────────

print("🧹 Cleaning all articles...\n")

cleaned_articles = []
cleaning_stats = {
    "total": len(parsed_articles),
    "empty_title_removed": 0,
    "empty_url_removed": 0,
    "cleaned": 0,
}

for article in parsed_articles:
    article = clean_article(article)
    
    if not article.title or article.title == "No Title":
        cleaning_stats["empty_title_removed"] += 1
        continue
    if not article.url:
        cleaning_stats["empty_url_removed"] += 1
        continue
    
    cleaned_articles.append(article)
    cleaning_stats["cleaned"] += 1

print(f"  Input articles:        {cleaning_stats['total']}")
print(f"  Removed (no title):    {cleaning_stats['empty_title_removed']}")
print(f"  Removed (no URL):      {cleaning_stats['empty_url_removed']}")
print(f"  Clean articles:        {cleaning_stats['cleaned']}")

🧹 Cleaning all articles...

  Input articles:        140
  Removed (no title):    0
  Removed (no URL):      0
  Clean articles:        140


In [13]:
# ─────────────────────────────────────────────────
#  DEDUPLICATION HELPERS
# ─────────────────────────────────────────────────

def make_url_fingerprint(url: str) -> str:
    if not url:
        return ""
    normalized = url.lower().strip().rstrip("/")
    normalized = re.sub(r'://www\.', '://', normalized)
    normalized = re.sub(r'#.*$', '', normalized)
    return hashlib.md5(normalized.encode()).hexdigest()


def make_title_fingerprint(title: str) -> str:
    if not title:
        return ""
    text = title.lower().strip()
    prefixes = [
        r'^breaking:\s*', r'^update:\s*', r'^exclusive:\s*',
        r'^just in:\s*', r'^live:\s*', r'^watch:\s*',
        r'^opinion:\s*', r'^analysis:\s*',
    ]
    for prefix in prefixes:
        text = re.sub(prefix, '', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = sorted(text.split())
    normalized = " ".join(words)
    return hashlib.md5(normalized.encode()).hexdigest()


def jaccard_similarity(text1: str, text2: str) -> float:
    def tokenize(text):
        text = re.sub(r'[^\w\s]', '', text.lower())
        stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'in', 'on',
                      'at', 'to', 'for', 'of', 'with', 'by', 'and', 'or', 'but',
                      'it', 'its', 'this', 'that', 'as', 'be', 'has', 'have'}
        return set(text.split()) - stop_words
    
    set1 = tokenize(text1)
    set2 = tokenize(text2)
    if not set1 or not set2:
        return 0.0
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0.0


def deduplicate_articles(articles, fuzzy_threshold=0.45, verbose=True):
    """Deduplicate using 3-layer approach: URL → title fingerprint → fuzzy match."""
    stats = {
        "input_count": len(articles),
        "url_dupes": 0, "title_dupes": 0, "fuzzy_dupes": 0,
        "output_count": 0, "duplicates_log": [],
    }
    
    # Layer 1: Exact URL
    if verbose:
        print("  Layer 1: Exact URL matching...")
    seen_urls = set()
    after_url = []
    for article in articles:
        url_fp = make_url_fingerprint(article.url)
        if url_fp in seen_urls:
            stats["url_dupes"] += 1
            continue
        seen_urls.add(url_fp)
        after_url.append(article)
    if verbose:
        print(f"    Removed {stats['url_dupes']} URL duplicates → {len(after_url)} remain")
    
    # Layer 2: Title fingerprint
    if verbose:
        print("  Layer 2: Title fingerprint matching...")
    seen_titles = set()
    after_title = []
    for article in after_url:
        title_fp = make_title_fingerprint(article.title)
        if title_fp in seen_titles:
            stats["title_dupes"] += 1
            continue
        seen_titles.add(title_fp)
        after_title.append(article)
    if verbose:
        print(f"    Removed {stats['title_dupes']} title duplicates → {len(after_title)} remain")
    
    # Layer 3: Fuzzy
    if verbose:
        print(f"  Layer 3: Fuzzy matching (threshold={fuzzy_threshold})...")
    unique = []
    for article in after_title:
        is_dup = False
        for existing in unique:
            if jaccard_similarity(article.title, existing.title) >= fuzzy_threshold:
                is_dup = True
                stats["fuzzy_dupes"] += 1
                break
        if not is_dup:
            unique.append(article)
    if verbose:
        print(f"    Removed {stats['fuzzy_dupes']} fuzzy duplicates → {len(unique)} remain")
    
    stats["output_count"] = len(unique)
    return unique, stats


print("✅ Deduplication engine defined.")

✅ Deduplication engine defined.


In [14]:
# ─────────────────────────────────────────────────
#  RUN DEDUPLICATION
# ─────────────────────────────────────────────────

print("🔄 Running deduplication pipeline...\n")

unique_articles, dedup_stats = deduplicate_articles(
    cleaned_articles,
    fuzzy_threshold=0.45,
    verbose=True,
)

print("\n" + "=" * 60)
print("  DEDUPLICATION REPORT")
print("=" * 60)
print(f"  Input articles:       {dedup_stats['input_count']}")
print(f"  URL duplicates:       {dedup_stats['url_dupes']}")
print(f"  Title duplicates:     {dedup_stats['title_dupes']}")
print(f"  Fuzzy duplicates:     {dedup_stats['fuzzy_dupes']}")
print(f"  ─────────────────────")
total_removed = dedup_stats['url_dupes'] + dedup_stats['title_dupes'] + dedup_stats['fuzzy_dupes']
print(f"  Total removed:        {total_removed}")
print(f"  Unique articles:      {dedup_stats['output_count']}")

🔄 Running deduplication pipeline...

  Layer 1: Exact URL matching...
    Removed 0 URL duplicates → 140 remain
  Layer 2: Title fingerprint matching...
    Removed 0 title duplicates → 140 remain
  Layer 3: Fuzzy matching (threshold=0.45)...
    Removed 3 fuzzy duplicates → 137 remain

  DEDUPLICATION REPORT
  Input articles:       140
  URL duplicates:       0
  Title duplicates:     0
  Fuzzy duplicates:     3
  ─────────────────────
  Total removed:        3
  Unique articles:      137


In [15]:
# ─────────────────────────────────────────────────
#  OLLAMA HELPER: Call the local LLM
# ─────────────────────────────────────────────────

def ollama_generate(prompt: str, model: str = OLLAMA_MODEL, timeout: int = 120) -> str:
    """Send a prompt to Ollama and return the response text."""
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.3,
            "num_predict": 300,
        }
    }
    
    try:
        resp = requests.post(url, json=payload, timeout=timeout)
        resp.raise_for_status()
        return resp.json().get("response", "").strip()
    except requests.exceptions.ConnectionError:
        print("❌ ERROR: Cannot connect to Ollama. Make sure 'ollama serve' is running!")
        return ""
    except Exception as e:
        print(f"❌ Ollama error: {e}")
        return ""


# ── Test connection ──
print("🔌 Testing Ollama connection...")
try:
    r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"   ✅ Ollama is running! Available models: {models}")
    if not any(OLLAMA_MODEL in m for m in models):
        print(f"   ⚠️  Model '{OLLAMA_MODEL}' not found. Run: ollama pull {OLLAMA_MODEL}")
except Exception as e:
    print(f"   ❌ Cannot reach Ollama at {OLLAMA_BASE_URL}: {e}")
    print(f"   Please start Ollama with: ollama serve")

🔌 Testing Ollama connection...
   ✅ Ollama is running! Available models: ['llama3:latest', 'qwen3.5:0.8b']


In [16]:
# ─────────────────────────────────────────────────
#  AI AGENT: Score & Summarize each article
# ─────────────────────────────────────────────────

def score_article(article: Article) -> dict:
    """
    Use Ollama to score an article's importance and generate a summary.
    Returns dict with 'score' (1-10) and 'summary' (one line).
    """
    prompt = f"""/no_think
You are a news editor AI. Rate this news article and provide a brief summary.

Title: {article.title}
Source: {article.source_name}
Description: {article.description[:300]}

Respond ONLY in this exact format (no extra text):
SCORE: <number 1-10>
SUMMARY: <one sentence summary>

Scoring guide:
- 9-10: Major breaking news, global impact
- 7-8: Important national/industry news
- 5-6: Notable but not urgent
- 3-4: Niche or routine updates
- 1-2: Fluff, clickbait, or trivial
"""

    response = ollama_generate(prompt)
    
    # Parse the response
    score = 5  # default
    summary = article.title  # fallback
    
    if response:
        for line in response.split("\n"):
            line = line.strip()
            if line.upper().startswith("SCORE:"):
                try:
                    score_text = line.split(":", 1)[1].strip()
                    # Extract first number found
                    nums = re.findall(r'\d+', score_text)
                    if nums:
                        score = min(max(int(nums[0]), 1), 10)
                except (ValueError, IndexError):
                    pass
            elif line.upper().startswith("SUMMARY:"):
                summary = line.split(":", 1)[1].strip()
                if not summary:
                    summary = article.title
    
    return {"score": score, "summary": summary}


print("✅ AI scoring function defined.")

✅ AI scoring function defined.


In [17]:
# ─────────────────────────────────────────────────
#  RUN AI SCORING on all unique articles
# ─────────────────────────────────────────────────

print(f"🤖 Scoring {len(unique_articles)} articles with {OLLAMA_MODEL}...\n")
print("   This may take a few minutes depending on your hardware.\n")

scored_articles = []

for i, article in enumerate(unique_articles, 1):
    result = score_article(article)
    scored_articles.append({
        "article": article,
        "score": result["score"],
        "summary": result["summary"],
    })
    
    # Progress indicator
    if i % 10 == 0 or i == len(unique_articles):
        print(f"   Processed {i}/{len(unique_articles)} articles...")

# Sort by score (highest first)
scored_articles.sort(key=lambda x: x["score"], reverse=True)

# Get top N
top_articles = scored_articles[:TOP_N_NEWS]

print(f"\n✅ Scoring complete! Top {TOP_N_NEWS} articles selected.\n")
print("=" * 70)
print(f"  🏆 TOP {TOP_N_NEWS} NEWS STORIES")
print("=" * 70)
for i, item in enumerate(top_articles, 1):
    a = item["article"]
    print(f"\n  {i:2d}. [Score: {item['score']}/10] {a.title[:70]}")
    print(f"      Source: {a.source_name}")
    print(f"      Summary: {item['summary'][:100]}")

🤖 Scoring 137 articles with qwen3.5:0.8b...

   This may take a few minutes depending on your hardware.

   Processed 10/137 articles...
   Processed 20/137 articles...
   Processed 30/137 articles...
   Processed 40/137 articles...
   Processed 50/137 articles...
   Processed 60/137 articles...
   Processed 70/137 articles...
   Processed 80/137 articles...
   Processed 90/137 articles...
   Processed 100/137 articles...
   Processed 110/137 articles...
   Processed 120/137 articles...
   Processed 130/137 articles...
   Processed 137/137 articles...

✅ Scoring complete! Top 10 articles selected.

  🏆 TOP 10 NEWS STORIES

   1. [Score: 5/10] Trump announces deal for Russian diesel as Zelensky calls it a gift to
      Source: BBC News - Top Stories
      Summary: Trump announces deal for Russian diesel as Zelensky calls it a gift to Putin

   2. [Score: 5/10] 'Cockroach' group leaders among hundreds detained in Delhi protest
      Source: BBC News - Top Stories
      Summary: 'Cockroac

In [18]:
# ─────────────────────────────────────────────────
#  BUILD EMAIL CONTENT
# ─────────────────────────────────────────────────

def build_email_html(top_articles: list, run_time: str) -> str:
    """Build a styled HTML email with the top news stories."""
    
    rows = ""
    for i, item in enumerate(top_articles, 1):
        a = item["article"]
        pub = a.published[:10] if a.published else "N/A"
        
        # Score badge color
        score = item["score"]
        if score >= 8:
            badge_color = "#22c55e"  # green
        elif score >= 6:
            badge_color = "#f59e0b"  # amber
        else:
            badge_color = "#94a3b8"  # gray
        
        rows += f"""
        <tr style="border-bottom: 1px solid #e2e8f0;">
            <td style="padding: 16px 12px; text-align: center; vertical-align: top;">
                <span style="background: {badge_color}; color: white; padding: 4px 10px;
                             border-radius: 12px; font-weight: bold; font-size: 14px;">
                    {score}/10
                </span>
            </td>
            <td style="padding: 16px 12px;">
                <a href="{a.url}" style="color: #1e40af; text-decoration: none;
                          font-size: 16px; font-weight: 600; line-height: 1.4;">
                    {i}. {a.title}
                </a>
                <br>
                <span style="color: #64748b; font-size: 13px;">
                    📰 {a.source_name} &nbsp;|&nbsp; 📅 {pub}
                </span>
                <br>
                <span style="color: #334155; font-size: 14px; line-height: 1.5;">
                    {item['summary'][:200]}
                </span>
            </td>
        </tr>
        """
    
    html = f"""
    <html>
    <body style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                 background: #f8fafc; margin: 0; padding: 20px;">
        <div style="max-width: 700px; margin: 0 auto; background: white;
                    border-radius: 12px; overflow: hidden;
                    box-shadow: 0 4px 6px rgba(0,0,0,0.07);">
            
            <!-- Header -->
            <div style="background: linear-gradient(135deg, #1e3a5f, #2563eb);
                        padding: 30px; text-align: center;">
                <h1 style="color: white; margin: 0; font-size: 26px;">
                    📰 AI News Digest
                </h1>
                <p style="color: #93c5fd; margin: 8px 0 0; font-size: 14px;">
                    Top {len(top_articles)} Stories — Ranked by AI ({OLLAMA_MODEL})
                </p>
                <p style="color: #93c5fd; margin: 4px 0 0; font-size: 12px;">
                    Generated: {run_time}
                </p>
            </div>
            
            <!-- Articles Table -->
            <table style="width: 100%; border-collapse: collapse;">
                {rows}
            </table>
            
            <!-- Footer -->
            <div style="background: #f1f5f9; padding: 20px; text-align: center;
                        font-size: 12px; color: #94a3b8;">
                🤖 Powered by AI News Agent | Ollama + {OLLAMA_MODEL}
                <br>RSS sources: {len(RSS_SOURCES)} feeds
            </div>
        </div>
    </body>
    </html>
    """
    return html


run_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
email_html = build_email_html(top_articles, run_time)
print("✅ Email HTML content built.")
print(f"   Content length: {len(email_html)} characters")

✅ Email HTML content built.
   Content length: 13700 characters


In [19]:
# ─────────────────────────────────────────────────
#  SEND EMAIL VIA GMAIL SMTP
# ─────────────────────────────────────────────────

def send_gmail(sender: str, password: str, recipient: str, subject: str, html_body: str) -> bool:
    """Send an HTML email via Gmail SMTP with TLS."""
    
    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"] = f"AI News Agent <{sender}>"
    msg["To"] = recipient
    
    # Plain text fallback
    plain_text = f"AI News Digest - Top {TOP_N_NEWS} Stories\n\n"
    for i, item in enumerate(top_articles, 1):
        a = item["article"]
        plain_text += f"{i}. [{item['score']}/10] {a.title}\n"
        plain_text += f"   {item['summary']}\n"
        plain_text += f"   {a.url}\n\n"
    
    msg.attach(MIMEText(plain_text, "plain"))
    msg.attach(MIMEText(html_body, "html"))
    
    try:
        print("📧 Connecting to Gmail SMTP...")
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.ehlo()
        server.starttls()
        server.ehlo()
        
        print("🔑 Authenticating...")
        server.login(sender, password)
        
        print("📤 Sending email...")
        server.sendmail(sender, recipient, msg.as_string())
        server.quit()
        
        print(f"\n✅ Email sent successfully to {recipient}!")
        return True
        
    except smtplib.SMTPAuthenticationError:
        print("\n❌ AUTHENTICATION FAILED!")
        print("   Make sure you're using a Gmail App Password, not your regular password.")
        print("   Get one at: https://myaccount.google.com/apppasswords")
        return False
    except Exception as e:
        print(f"\n❌ Failed to send email: {e}")
        return False


# ── Send the digest! ──
subject = f"📰 AI News Digest — Top {TOP_N_NEWS} Stories ({datetime.now().strftime('%b %d, %Y')})"

send_gmail(
    sender=GMAIL_ADDRESS,
    password=GMAIL_APP_PASSWORD,
    recipient=RECIPIENT_EMAIL,
    subject=subject,
    html_body=email_html,
)

📧 Connecting to Gmail SMTP...
🔑 Authenticating...
📤 Sending email...

✅ Email sent successfully to sankanoort@gmail.com!


True